In [5]:
using Random

using XLSX
using DataFrames

# using DifferentialEquations
# using DiffEqFlux
using SciMLSensitivity
using Flux

using ProgressMeter

include("../src/utils.jl");

In [6]:
file_path = "../data/STEMI_merged.xlsx";
sheet_times = "Tempi cleaned";
sheet_values = "Misurazioni cleaned";

In [7]:
times_df = DataFrame(XLSX.readtable(file_path, sheet_times, "A:X", header=false, infer_eltypes=true));
ids = times_df[:, :A]
select!(times_df, Not(:A))
values_df = DataFrame(XLSX.readtable(file_path, sheet_values, "B:X", header=false, infer_eltypes=true));

In [4]:
println("Patients: ", nrow(values_df))

Patients: 125


In [5]:
# --- Split dei dati ---
train_idx, val_idx, test_idx = split_data(nrow(values_df))

([125, 50, 112, 65, 113, 29, 121, 79, 5, 66  …  80, 84, 118, 97, 81, 108, 122, 58, 31, 48], [82, 23, 64, 55, 30, 39, 61, 74, 100, 94, 19, 17, 60, 59, 24, 7, 32, 117, 3], [92, 102, 67, 4, 41, 89, 109, 43, 49, 90, 42, 70, 88, 34, 116, 15, 44, 105])

In [6]:
# Splitting
train_ids = ids[train_idx, :]
train_times = times_df[train_idx, :]
train_values = values_df[train_idx, :]

val_ids = ids[val_idx, :]
val_times = times_df[val_idx, :]
val_values = values_df[val_idx, :]

test_ids = ids[test_idx, :]
test_times = times_df[test_idx, :]
test_values = values_df[test_idx, :]

Row,B,C,D,E,F,G,H,I,J,K,L,M,N,O,P,Q,R,S,T,U,V,W,X
,Float64,Float64,Float64,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?
1,0.1113,0.3889,0.5794,0.5239,0.5187,0.3752,0.1949,0.0878,0.0405,0.0205,0.0136,0.0081,0.0067,0.0067,0.0059,0.0059,0.0072,missing,missing,missing,missing,missing,missing
2,0.596,5.22,11.77,9.26,7.76,6.64,4.02,1.16,0.569,0.325,0.143,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing
3,0.153,0.155,0.174,0.113,0.114,0.142,0.07,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing
4,0.44,5.71,5.0,4.2,3.6,3.94,4.04,4.33,4.08,2.84,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing
5,0.058,0.215,0.331,0.298,0.33,0.333,0.196,0.078,0.035,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing
6,0.426,2.38,1.88,2.37,2.45,1.69,0.71,0.273,0.147,0.047,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing
7,2.09,6.96,5.99,7.67,9.27,7.56,3.29,1.71,0.657,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing
8,1.68,3.74,6.42,3.69,3.05,2.41,2.14,2.39,1.48,0.62,0.319,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing
9,0.206,0.803,1.05,0.504,0.753,0.62,0.428,0.247,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing


In [7]:
# Definizione della rete neurale che fornisce la correzione
# Input: lo stato u e il tempo t (si potrebbe concatenare t ad u)
neural_net = Flux.Chain(
    Flux.Dense(4, 16, relu),   # ipotizzando un input di dimensione 4 (o 3+1 se includi t)
    Flux.Dense(16, 1, softplus)          # output: correzione per i primi 2 stati
)

# NN parameters
nn_params = Flux.trainable(neural_net)

(layers = (Dense(4 => 16, relu), Dense(16 => 1, softplus)),)

In [8]:
"""
    ctnt_ode!(du, u, p, t)

Compartments. 
- u[1]: sarcomere
- u[2]: citosol
- u[3]: plasma
- p: model parameters
"""
function ctnt_ode!(du, u, p, t)
    # Esempio di termini dinamici (da adattare al modello specifico)
    # Termini base (senza correzione)
    # p = [a, b, Cs0, Cc0]
    # Cs_ctnt = u[1]
    # Cc_ctnt = u[2]
    # Cp_ctnt = u[3]

    du[1] = - (u[1] - u[2])
    du[2] = u[1] - u[2] - p[1]*(u[2] - u[3])
    du[3] = p[1]*(u[2] - u[3]) - p[2]*u[3]
    
    # Calcolo della correzione tramite la rete neurale per i primi due stati
    # Si può usare, ad esempio, u_corr = neural_net(vcat(u[1:3], [t]))
    # (Adattare la dimensione dell’input in base al progetto)
    correction = neural_net(vcat(u[1:3], [t]))[1]
    
    # Aggiungi la correzione solo agli stati 1 e 2
    du[1] += correction
    du[2] += correction
    
    # Stato 3 (plasma) rimane invariato dalla correzione
end

# Funzione per simulare l'ODE per un dato paziente
function simulate_ctnt(u0, p, tspan)
    println("Simulating model")
    prob = ODEProblem(ctnt_ode!, u0, tspan, p)
    sol = solve(prob, Tsit5(), sensealg=SensitivityADPassThrough())
    return sol
end

simulate_ctnt (generic function with 1 method)

In [9]:
neural_net(vcat([10,20,30], [0]))[1]

11.052574f0

In [10]:
function loss_patient(i, t_data, observed, patient_params)
    # Filtro: mantengo solo le coppie (tempo, valore) non missing
    println("Calculating patient loss")
    valid_idx = findall(.!ismissing.(t_data) .& .!ismissing.(observed))
    t_data = t_data[valid_idx]
    observed = observed[valid_idx]
    if isempty(t_data)
        return 0.0
    end

    # Stato iniziale: gli stati 1 e 2 a zero e lo stato 3 uguale al primo valore osservato
    p = patient_params
    u0 = [p[3], p[4], 0.0]
      
    tspan = (minimum(t_data), maximum(t_data))
    
    sol = simulate_ctnt(u0, p, tspan)
    # Interpoliamo la soluzione sui tempi in cui sono state fatte le misurazioni
    pred = sol.(t_data)
    # Estraiamo il compartimento plasma (stato 3)
    pred_plasma = [u[3] for u in pred]
    
    # Errore quadratico tra predizioni e dati osservati
    return sum((pred_plasma .- observed).^2)
end

loss_patient (generic function with 1 method)

In [11]:
# Loss totale: somma delle loss su tutti i pazienti del training set
function total_loss(train_times, train_values, patients_params)
    println("Calculating total loss")
    loss_total = 0.0
    for i in 1:nrow(train_times) 
        t_data   = collect(skipmissing(train_times[i, :]))
        observed = collect(skipmissing(train_values[i, :]))
        loss_total += loss_patient(i, t_data, observed, patients_params[i])
    end
    return loss_total
end

total_loss (generic function with 1 method)

Training phase

In [12]:
function training_UDE!(model, n_epochs, ode_init_params, n_samples, optimizer)
    # Per ogni paziente definiamo un vettore trainabile di 4 parametri, inizializzati a valori di default.
    patient_params = [ode_init_params for _ in 1:n_samples]
    # params_all = Flux.trainable(neural_net) ∪ Flux.trainable(patient_params)
    # params_model = Flux.trainable(model)
    # params_patient = Flux.trainable(patient_params)

    params_all = Flux.params(neural_net, patient_params)

    println("Starting params defined")
    
    # params_all = vcat(params_model, params_patient)

    # Inizializza i parametri per il salvataggio del modello migliore
    best_loss = Inf
    best_model = nothing

    @showprogress for epoch in 1:n_epochs
        # Calcolo e aggiornamento del training loss
        println("Training epoch: $epoch")
        grads = gradient(params_all) do
            total_loss(train_times, train_values, patient_params)
        end
        Flux.Optimise.update!(opt, params_all, grads)
        
        # Calcolo della loss sul validation set
        val_loss = total_loss(val_times, val_values)
        
        println("Epoch $epoch: Training Loss = ", total_loss(train_times, train_values),
                " | Validation Loss = ", val_loss)
        
        # Salva il modello se il validation loss migliora
        if val_loss < best_loss
            best_loss = val_loss
            best_model = deepcopy(params_all)  # Salva una copia dei parametri
            println("New best model!")
        end
    end
    return best_loss, best_model, params_all
end

training_UDE! (generic function with 1 method)

In [13]:
# model = neural_net
n_epochs = 100
params_init = [0.005, 0.005, 0.1, 0.001] # ODE params
n_patients = length(train_idx) 
opt = ADAM(0.01)

best_loss, best_model, params_all = training_UDE!(neural_net, n_epochs, params_init, n_patients, opt)

Starting params defined
Training epoch: 1


┌ Warning: `Flux.params(m...)` is deprecated. Use `Flux.trainable(model)` for parameter collection,
│ and the explicit `gradient(m -> loss(m, x, y), model)` for gradient computation.
└ @ Flux C:\Users\nicol\.julia\packages\Flux\BkG8S\src\deprecations.jl:93
┌ Warning: Implicit gradients such as `gradient(f, ::Params)` are deprecated in Flux!
│ Please see the docs for new explicit form.
│   caller = macro expansion at jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X15sZmlsZQ==.jl:21 [inlined]
└ @ Core c:\Users\nicol\PHD_Projects\MechanisticAI\MechanisticsAI_julia\notebooks\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X15sZmlsZQ==.jl:21


Calculating total loss
Calculating patient loss
Simulating model


Zygote.CompileError: Compiling Tuple{typeof(FunctionWrappers.gen_fptr), Type{Nothing}, Type{Tuple{Vector{Float64}, Vector{Float64}, Vector{Float64}, Float64}}, Type{SciMLBase.Void{typeof(ctnt_ode!)}}}: UndefVarError: `spvals` not defined in `Base.Meta`
Suggestion: check for spelling errors or missing imports.

In [14]:
# 6. Fase di test per un nuovo paziente
# ===============================
# Una volta addestrata la rete, i pesi della rete sono validi in generale.
# Per un nuovo paziente:
#  - La rete neurale rimane fissa.
#  - I parametri specifici devono essere calibrati localmente.
#
# Supponiamo di avere i dati per un nuovo paziente:
new_t_data   = rand(10)
new_observed = rand(10)  # misurazioni del compartimento plasma per il nuovo paziente

# Inizializziamo i parametri del nuovo paziente (che saranno ottimizzati localmente)
new_patient_param = Flux.trainable([1.0, 0.5, 0.3, 0.1])
tspan_new = (minimum(new_t_data), maximum(new_t_data))
u0_new    = [0.0, 0.0, new_observed[1]]

# Loss per il nuovo paziente con rete fissa e parametri ottimizzabili localmente
function loss_new_patient(new_patient_param, t_data, observed)
    sol = simulate_ctnt(u0_new, new_patient_param, tspan_new)
    pred = sol.(t_data)
    pred_plasma = [u[3] for u in pred]
    return sum((pred_plasma .- observed).^2)
end

# Ottimizziamo i parametri specifici per il nuovo paziente (con rete neurale fissata)
local_opt = ADAM(0.01)
n_local_epochs = 20
for epoch in 1:n_local_epochs
    grads = gradient(() -> loss_new_patient(new_patient_param, new_t_data, new_observed), Flux.trainable(new_patient_param))
    Flux.Optimise.update!(local_opt, Flux.trainable(new_patient_param), grads)
    println("Local Epoch $epoch: Loss = $(loss_new_patient(new_patient_param, new_t_data, new_observed))")
end

println("Test completato: la rete neurale è fissa e i parametri del nuovo paziente sono stati calibrati localmente.")

MethodError: MethodError: no method matching (::var"#20#21")(::Tuple{})
The function `#20` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  (::var"#20#21")()
   @ Main c:\Users\nicol\PHD_Projects\MechanisticAI\MechanisticsAI_julia\notebooks\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X16sZmlsZQ==.jl:29
